# Cadeias de Markov e regularidade do consumo de proteína

**Pergunta:** o estado de hoje (abaixo / dentro / acima da faixa da meta de proteína) ajuda a prever o estado de amanhã?

Base: MyFitnessPal Food Diary Dataset (Weber e Achananuparp). O notebook executa do início ao fim e segue as etapas:
1. Preparação · 2. Estados · 3. Transições · 4. Previsões · 5. Avaliação · 6. Resultados.

**Convenção da matriz de transição:** `P[i, j]` = probabilidade de ir para o estado `i` amanhã, dado que hoje o estado é `j`.
Cada **coluna** representa o estado de origem e soma 1, então `p_amanhã = P @ p_hoje`.

Todos os parâmetros metodológicos ficam na célula de configuração abaixo, definidos **antes** de qualquer avaliação.

In [ ]:
import csv
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    display
except NameError:  # execução fora do Jupyter
    display = print

# ----------------------------------------------------------------------------
# CONFIGURAÇÃO (decisões metodológicas, fixadas antes da avaliação)
# ----------------------------------------------------------------------------
DATA_PATH = Path(os.environ.get("MFP_TSV", "dados/mfp-diaries.tsv"))  # base bruta (fora do Git)
OUT_DIR = Path("resultados")
CHUNKSIZE = 50_000

USER_COL, DATE_COL = 0, 1       # posições das colunas de pessoa e data no TSV (sem cabeçalho)

# Faixa de tolerância em relação à meta diária de proteína: razão = consumo / meta
BAND_LOW, BAND_HIGH = 0.90, 1.10   # abaixo: razão < 0,90 | dentro: 0,90–1,10 | acima: razão > 1,10

# Filtros de qualidade
GOAL_MIN, GOAL_MAX = 20.0, 400.0     # meta de proteína plausível (g/dia)
PROT_MAX = 500.0                     # consumo de proteína plausível (g/dia); precisa ser > 0
KCAL_MIN, KCAL_MAX = 800.0, 6000.0   # dias com calorias fora disso são considerados registro incompleto/implausível
MIN_DAYS_USER = 14                   # mínimo de dias válidos por pessoa para entrar no recorte

# Avaliação
TRAIN_FRAC = 0.60     # fração do intervalo de datas usada para treino (corte cronológico)
MIN_COL_OBS = 30      # mínimo de transições de uma origem (no treino) para confiar na coluna da matriz
N_BOOT, SEED = 1000, 42

STATES = ["Abaixo", "Dentro", "Acima"]
K = len(STATES)
OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Preparação

O TSV não tem cabeçalho: colunas de pessoa e data, o JSON das refeições e o(s) JSON(s) com `total` e `goal` do dia.
A leitura é feita em blocos e **apenas** as colunas necessárias são interpretadas (o JSON das refeições, que é o mais pesado, é ignorado).
Os valores dos JSON podem vir como dicionário ou como lista de `{name, value}`, com vírgulas de milhar (`"1,640"`); o parser aceita as duas formas.

In [ ]:
def para_numero(x):
    """Converte '1,640', '90 g', 90 etc. em float (NaN se não der)."""
    if x is None:
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = re.sub(r"[^0-9.\-]", "", str(x).replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def dicionario_nutrientes(obj):
    """Normaliza {'Protein': '90'} ou [{'name': 'Protein', 'value': '90'}] em {nome_minusculo: float}."""
    saida = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if not isinstance(v, (dict, list)):
                saida[str(k).strip().lower()] = para_numero(v)
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict) and "name" in item and "value" in item:
                saida[str(item["name"]).strip().lower()] = para_numero(item["value"])
    return saida


def pega(d, prefixo):
    """Primeiro valor cuja chave começa com `prefixo` (ex.: 'protein', 'calories')."""
    for k, v in d.items():
        if k.startswith(prefixo):
            return v
    return np.nan


def extrai_linha(textos):
    """Recebe os textos JSON de uma linha e devolve (proteína consumida, meta de proteína, calorias)."""
    total, meta = {}, {}
    for t in textos:
        if not isinstance(t, str):
            continue
        try:
            obj = json.loads(t)
        except ValueError:
            continue
        if isinstance(obj, dict):
            if "total" in obj:
                total.update(dicionario_nutrientes(obj["total"]))
            if "goal" in obj:
                meta.update(dicionario_nutrientes(obj["goal"]))
    return pega(total, "protein"), pega(meta, "protein"), pega(total, "calories")


def detecta_layout(caminho):
    """Descobre se há cabeçalho e quais colunas trazem os JSON de total/goal."""
    amostra = pd.read_csv(caminho, sep="\t", header=None, dtype=str, nrows=300,
                          quoting=csv.QUOTE_NONE, on_bad_lines="skip")
    tem_cabecalho = bool(pd.isna(pd.to_datetime(amostra.iloc[0, DATE_COL], errors="coerce")))
    if tem_cabecalho:
        amostra = amostra.iloc[1:]
    cols_json = [c for c in amostra.columns
                 if c not in (USER_COL, DATE_COL)
                 and amostra[c].dropna().astype(str).str.contains('"goal"|"total"', regex=True).any()]
    if not cols_json:
        raise ValueError("Nenhuma coluna com JSON de 'total'/'goal' encontrada. Amostra:\n" + amostra.head(2).to_string()[:800])
    return tem_cabecalho, cols_json


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Base não encontrada em {DATA_PATH}. Baixe o TSV do Kaggle e coloque em dados/ "
        "(ou defina a variável de ambiente MFP_TSV). Veja o README.")

tem_cabecalho, cols_json = detecta_layout(DATA_PATH)
print(f"Cabeçalho: {tem_cabecalho} | colunas com JSON de total/goal: {cols_json}")

partes, n_lidos = [], 0
leitor = pd.read_csv(DATA_PATH, sep="\t", header=None, skiprows=1 if tem_cabecalho else 0,
                     usecols=[USER_COL, DATE_COL] + cols_json, dtype=str,
                     quoting=csv.QUOTE_NONE, on_bad_lines="skip", chunksize=CHUNKSIZE)
for bloco in leitor:
    n_lidos += len(bloco)
    vals = [extrai_linha(r) for r in bloco[cols_json].itertuples(index=False, name=None)]
    partes.append(pd.DataFrame({
        "user_id": bloco[USER_COL].to_numpy(),
        "date": pd.to_datetime(bloco[DATE_COL], errors="coerce").dt.normalize().to_numpy(),
        "proteina": [v[0] for v in vals],
        "meta": [v[1] for v in vals],
        "kcal": [v[2] for v in vals],
    }))
bruto = pd.concat(partes, ignore_index=True)
del partes
print(f"{n_lidos:,} linhas lidas".replace(",", "."))

### Filtros (cada um registra quantos registros removeu)

In [ ]:
filtros = [{"filtro": "registros lidos", "removidos": 0, "restantes": len(bruto)}]


def aplica(df, nome, manter):
    saida = df[manter]
    filtros.append({"filtro": nome, "removidos": len(df) - len(saida), "restantes": len(saida)})
    return saida


df = bruto
df = aplica(df, "data inválida", df["date"].notna())
df = aplica(df, "proteína ou meta ausente", df["proteina"].notna() & df["meta"].notna())
df = aplica(df, f"meta fora de [{GOAL_MIN:g}, {GOAL_MAX:g}] g", df["meta"].between(GOAL_MIN, GOAL_MAX))
df = aplica(df, f"proteína ≤ 0 ou > {PROT_MAX:g} g", (df["proteina"] > 0) & (df["proteina"] <= PROT_MAX))
df = aplica(df, f"calorias fora de [{KCAL_MIN:g}, {KCAL_MAX:g}] (registro incompleto)",
            df["kcal"].isna() | df["kcal"].between(KCAL_MIN, KCAL_MAX))
df = aplica(df, "duplicata (pessoa, data)", ~df.duplicated(["user_id", "date"], keep="first"))
dias_por_pessoa = df.groupby("user_id")["date"].transform("size")
df = aplica(df, f"pessoas com menos de {MIN_DAYS_USER} dias válidos", dias_por_pessoa >= MIN_DAYS_USER)

tabela_filtros = pd.DataFrame(filtros)
tabela_filtros.to_csv(OUT_DIR / "filtros.csv", index=False)
display(tabela_filtros)

df = df.sort_values(["user_id", "date"]).reset_index(drop=True)
print(f"Aproveitados: {len(df):,} dias de {df['user_id'].nunique():,} pessoas "
      f"({df['date'].min().date()} a {df['date'].max().date()})".replace(",", "."))

## 2. Estados

Razão `r = proteína consumida / meta`. Estado **Abaixo** se `r < BAND_LOW`, **Acima** se `r > BAND_HIGH`, **Dentro** caso contrário.
Códigos: 0 = Abaixo, 1 = Dentro, 2 = Acima.

In [ ]:
def classifica(razao):
    return np.where(razao < BAND_LOW, 0, np.where(razao > BAND_HIGH, 2, 1))


df["razao"] = df["proteina"] / df["meta"]
df["estado"] = classifica(df["razao"].to_numpy())

freq_geral = df["estado"].value_counts(normalize=True).reindex(range(K), fill_value=0.0)
print("Frequência geral dos estados (todos os dias aproveitados):")
display(pd.DataFrame({"estado": STATES, "dias": df["estado"].value_counts().reindex(range(K), fill_value=0).to_numpy(),
                      "frequencia": freq_geral.to_numpy()}))

# Sequência de estados de cada pessoa (amostra)
display(df.groupby("user_id")["estado"].apply(lambda s: "".join(map(str, s.head(30)))).head(5))

## 3. Transições

Só contam pares de dias **consecutivos (diferença de exatamente 1 dia) da mesma pessoa**. Intervalos maiores não viram transição.
`C[i, j]` = nº de vezes em que a pessoa estava em `j` num dia e em `i` no dia seguinte; `P = C` com cada coluna dividida pela sua soma.

In [ ]:
def pares_consecutivos(d):
    """Pares (dia t, dia t+1) da mesma pessoa, com exatamente 1 dia de diferença."""
    d = d.sort_values(["user_id", "date"])
    g = d.groupby("user_id", sort=False)
    prox_estado, prox_data = g["estado"].shift(-1), g["date"].shift(-1)
    ok = (prox_data - d["date"]).dt.days == 1
    return pd.DataFrame({
        "user_id": d.loc[ok, "user_id"].to_numpy(),
        "data_origem": d.loc[ok, "date"].to_numpy(),
        "data_destino": prox_data[ok].to_numpy(),
        "origem": d.loc[ok, "estado"].to_numpy().astype(int),
        "destino": prox_estado[ok].to_numpy().astype(int),
    })


def matriz_transicao(pares):
    """Devolve contagens C, probabilidades P (coluna sem observações = NaN) e nº de transições por origem."""
    C = np.zeros((K, K), dtype=int)
    np.add.at(C, (pares["destino"].to_numpy(), pares["origem"].to_numpy()), 1)
    n_col = C.sum(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        P = C / n_col
    return C, P, n_col


pares = pares_consecutivos(df)
assert ((pares["data_destino"] - pares["data_origem"]).dt.days == 1).all()
assert len(pares) <= len(df) - df["user_id"].nunique()   # nenhum par cruza pessoas

C, P, n_col = matriz_transicao(pares)
sem_obs = [STATES[j] for j in range(K) if n_col[j] == 0]
if sem_obs:
    print("ATENÇÃO: estados de origem sem transições observadas:", sem_obs)

# conferências: probabilidades não negativas e colunas válidas somando 1
validas = n_col > 0
assert (P[:, validas] >= 0).all() and np.allclose(P[:, validas].sum(axis=0), 1.0)

rot_dest = [f"amanhã: {s}" for s in STATES]
rot_orig = [f"hoje: {s}" for s in STATES]
tab_C = pd.DataFrame(C, index=rot_dest, columns=rot_orig)
tab_P = pd.DataFrame(P, index=rot_dest, columns=rot_orig)
tab_C.to_csv(OUT_DIR / "contagens_transicao.csv")
tab_P.to_csv(OUT_DIR / "matriz_transicao.csv")

resumo = pd.DataFrame({"item": ["pessoas", "dias aproveitados", "pares de dias consecutivos", "pares / dias"],
                       "valor": [df["user_id"].nunique(), len(df), len(pares), round(len(pares) / len(df), 3)]})
resumo.to_csv(OUT_DIR / "resumo_recorte.csv", index=False)
display(resumo)
print("Contagens C (colunas = origem):"); display(tab_C)
print("Matriz de transição P (cada coluna soma 1):"); display(tab_P.round(3))

## 4. Previsões

Com `p_hoje` o vetor de probabilidades dos estados hoje, `p_(t+k) = P^k @ p_hoje`. A coluna `j` de `P^k` dá a distribuição daqui a `k` dias partindo do estado `j`.
Depois, o autovetor de `P` associado ao autovalor 1 (normalizado para somar 1) dá a distribuição estacionária.
Nesta seção usa-se a matriz do recorte completo (descritivo); a avaliação preditiva (seção 5) usa só o treino.

In [ ]:
if not validas.all():
    print("Matriz incompleta: previsões e estacionária puladas (ver aviso da seção 3).")
else:
    linhas = []
    for k in [1, 2, 3, 7, 14, 30]:
        Pk = np.linalg.matrix_power(P, k)
        assert np.allclose(Pk.sum(axis=0), 1.0)
        for j in range(K):
            for i in range(K):
                linhas.append({"passos": k, "origem": STATES[j], "destino": STATES[i], "prob": Pk[i, j]})
    prev = pd.DataFrame(linhas)
    prev.to_csv(OUT_DIR / "previsoes_k_passos.csv", index=False)
    display(prev.pivot_table(index=["passos", "origem"], columns="destino", values="prob")[STATES].round(3))

    # Autovetor do autovalor 1
    autovalores, autovetores = np.linalg.eig(P)
    idx = int(np.argmin(np.abs(autovalores - 1)))
    pi = np.real(autovetores[:, idx])
    pi = pi / pi.sum()
    assert np.allclose(P @ pi, pi) and (pi >= -1e-12).all()

    modulos = np.sort(np.abs(autovalores))[::-1]
    regular = any((np.linalg.matrix_power(P, k) > 0).all() for k in range(1, (K - 1) ** 2 + 2))
    P_lim = np.linalg.matrix_power(P, 200)
    print(f"Autovalores (módulo): {np.round(modulos, 4)}  | |λ2| = {modulos[1]:.4f} (quanto menor, mais rápida a convergência)")
    print(f"Matriz regular (alguma potência estritamente positiva): {regular}")
    print(f"Maior diferença entre colunas de P^200 e a estacionária: {np.abs(P_lim - pi[:, None]).max():.2e}")

    estacionaria = pd.DataFrame({"estado": STATES, "estacionaria": pi,
                                 "frequencia_geral": freq_geral.to_numpy()})
    estacionaria.to_csv(OUT_DIR / "distribuicao_estacionaria.csv", index=False)
    display(estacionaria.round(4))

## 5. Avaliação (treino × teste por data)

**Protocolo:** corte cronológico em `TRAIN_FRAC` do intervalo de datas.
- Treino: pares cujo dia de destino é ≤ data de corte. A matriz `P_treino` e a frequência dos estados (`f_treino`, calculada nos dias de treino) usam **somente** estes dados.
- Teste: pares cujo dia de origem é > data de corte. Pares que atravessam o corte são descartados (sem vazamento).
- Regra para estados com poucos dados: se uma origem tem menos de `MIN_COL_OBS` transições no treino, a previsão Markov para essa origem usa `f_treino` (e isso é sinalizado).

**Métodos comparados:** (A) Markov: probabilidades = coluna da origem em `P_treino`; (B) frequência: sempre `f_treino`, ignorando o estado de hoje.
**Métricas:** acurácia (classe de maior probabilidade), erro de Brier (soma dos quadrados dos erros de probabilidade; menor é melhor) e log-loss.
Incerteza: bootstrap por pessoa (reamostra pessoas com reposição).

In [ ]:
data_corte = df["date"].min() + (df["date"].max() - df["date"].min()) * TRAIN_FRAC
data_corte = data_corte.normalize()
print("Data de corte:", data_corte.date())

df_treino = df[df["date"] <= data_corte]
pares_treino = pares[pares["data_destino"] <= data_corte]
pares_teste = pares[pares["data_origem"] > data_corte]
print(f"Treino: {len(df_treino):,} dias, {len(pares_treino):,} pares | Teste: {len(pares_teste):,} pares".replace(",", "."))
assert len(pares_treino) > 0 and len(pares_teste) > 0, "Treino ou teste vazio: ajuste TRAIN_FRAC/filtros."

# estimativas somente com o treino
f_treino = df_treino["estado"].value_counts(normalize=True).reindex(range(K), fill_value=0.0).to_numpy()
C_tr, P_tr, n_col_tr = matriz_transicao(pares_treino)
P_markov = P_tr.copy()
usa_fallback = n_col_tr < MIN_COL_OBS
for j in np.where(usa_fallback)[0]:
    P_markov[:, j] = f_treino
    print(f"Origem '{STATES[j]}' com {n_col_tr[j]} transições no treino (< {MIN_COL_OBS}): usando a frequência como previsão.")
assert (P_markov >= 0).all() and np.allclose(P_markov.sum(axis=0), 1.0)

# previsões no teste
y = pares_teste["destino"].to_numpy()
orig = pares_teste["origem"].to_numpy()
Y = np.eye(K)[y]
prob_markov = P_markov[:, orig].T                       # (n, K)
prob_freq = np.tile(f_treino, (len(y), 1))

EPS = 1e-12
def por_amostra(prob):
    acerto = (prob.argmax(axis=1) == y).astype(float)
    brier = ((prob - Y) ** 2).sum(axis=1)
    logloss = -np.log(np.clip(prob[np.arange(len(y)), y], EPS, 1))
    return acerto, brier, logloss

metricas = {"Markov": por_amostra(prob_markov), "Frequência (treino)": por_amostra(prob_freq)}
nomes_metricas = ["acuracia", "brier", "logloss"]

# bootstrap por pessoa
usuarios, inv = np.unique(pares_teste["user_id"].to_numpy(), return_inverse=True)
n_u = np.bincount(inv).astype(float)
soma_u = {m: [np.bincount(inv, weights=v) for v in vs] for m, vs in metricas.items()}
rng = np.random.default_rng(SEED)
boot = {m: np.empty((N_BOOT, 3)) for m in metricas}
diff = np.empty((N_BOOT, 3))
for b in range(N_BOOT):
    sel = rng.integers(0, len(usuarios), len(usuarios))
    den = n_u[sel].sum()
    for m in metricas:
        boot[m][b] = [s[sel].sum() / den for s in soma_u[m]]
    diff[b] = boot["Markov"][b] - boot["Frequência (treino)"][b]

linhas = []
for m, vs in metricas.items():
    for q, nome in enumerate(nomes_metricas):
        lo, hi = np.percentile(boot[m][:, q], [2.5, 97.5])
        linhas.append({"metodo": m, "metrica": nome, "valor": vs[q].mean(), "ic95_inf": lo, "ic95_sup": hi})
avaliacao = pd.DataFrame(linhas)
avaliacao.to_csv(OUT_DIR / "avaliacao.csv", index=False)

dif = pd.DataFrame([{"metrica": nome, "markov_menos_frequencia": metricas["Markov"][q].mean() - metricas["Frequência (treino)"][q].mean(),
                     "ic95_inf": np.percentile(diff[:, q], 2.5), "ic95_sup": np.percentile(diff[:, q], 97.5)}
                    for q, nome in enumerate(nomes_metricas)])
dif.to_csv(OUT_DIR / "avaliacao_diferenca.csv", index=False)

print(f"Pares no teste: {len(y):,} | pessoas no teste: {len(usuarios):,}".replace(",", "."))
display(avaliacao.pivot(index="metodo", columns="metrica", values="valor")[nomes_metricas].round(4))
print("Diferença Markov − frequência (acurácia: >0 é melhor; Brier e log-loss: <0 é melhor):")
display(dif.round(4))

pd.DataFrame({"origem": STATES, "transicoes_treino": n_col_tr, "usa_frequencia_no_lugar": usa_fallback}) \
    .to_csv(OUT_DIR / "treino_cobertura_origens.csv", index=False)
pd.DataFrame(P_markov, index=rot_dest, columns=rot_orig).to_csv(OUT_DIR / "matriz_transicao_treino.csv")

## 6. Resultados: figuras

Figura 1: matriz de transição (recorte completo). Figura 2: comparação Markov × frequência no teste, com IC 95% (bootstrap por pessoa).

In [ ]:
# Figura 1: matriz de transição
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(np.nan_to_num(P), cmap="Blues", vmin=0, vmax=1)
for i in range(K):
    for j in range(K):
        txt = "–" if np.isnan(P[i, j]) else f"{P[i, j]:.2f}"
        cor = "white" if np.nan_to_num(P[i, j]) > 0.55 else "black"
        ax.text(j, i, f"{txt}\n(n={C[i, j]:,})".replace(",", "."), ha="center", va="center", color=cor, fontsize=9)
ax.set_xticks(range(K), STATES)
ax.set_yticks(range(K), STATES)
ax.set_xlabel("Estado hoje (origem)")
ax.set_ylabel("Estado amanhã (destino)")
ax.set_title("Matriz de transição: proteína vs. meta")
fig.colorbar(im, ax=ax, label="Probabilidade", shrink=0.85)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig1_matriz_transicao.png", dpi=200)
plt.show()

In [ ]:
# Figura 2: avaliação
cores = {"Markov": "#1f6fb5", "Frequência (treino)": "#9a9a9a"}
rotulos = {"acuracia": "Acurácia (maior é melhor)", "brier": "Erro de Brier (menor é melhor)"}
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
for ax, nome in zip(axes, ["acuracia", "brier"]):
    for pos, m in enumerate(metricas):
        r = avaliacao[(avaliacao["metodo"] == m) & (avaliacao["metrica"] == nome)].iloc[0]
        ax.bar(pos, r["valor"], color=cores[m], width=0.6)
        ax.errorbar(pos, r["valor"], yerr=[[r["valor"] - r["ic95_inf"]], [r["ic95_sup"] - r["valor"]]],
                    color="black", capsize=4, lw=1.2)
        ax.text(pos, r["ic95_sup"], f"{r['valor']:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(len(metricas)), list(metricas))
    ax.set_title(rotulos[nome], fontsize=10)
    lim = avaliacao[avaliacao["metrica"] == nome]
    ax.set_ylim(0, lim["ic95_sup"].max() * 1.15)
fig.suptitle(f"Teste a partir de {(data_corte + pd.Timedelta(days=1)).date()} · barras = IC 95% (bootstrap por pessoa)", fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig2_avaliacao.png", dpi=200)
plt.show()

print("Arquivos gerados em", OUT_DIR.resolve())
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name)